In [1]:
GPT_CONFIG_124M ={
    "vocab_size" : 50257,
    "context_length" : 1024,
    "emb_dim" : 768,
    "n_heads":12,
    "n_layers":12,
    "drop_rate":0.1,
    "qkv_bias": False
}

In [2]:
import torch
import torch.nn as nn

class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])  
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks= nn.Sequential(
            *[DummyTransformerBlock(cfg)
            for _ in range(cfg["n_layers"])]    #hmm this is something new
        )
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])  # this is also something new
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias= False
        )

    def forward (self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device= in_idx.device)
        )
        x= tok_embeds + pos_embeds
        x= self.drop_emb(x)
        x= self.trf_blocks(x)
        x= self.final_norm(x)
        logits= self.out_head(x)
        return logits
    
class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

    def forward(self, x):
        return x
    
class DummyLayerNorm(nn.Module):
    def __init__ (self, normalised_shape, esp = 1e-5):
        super().__init__()

    def forward(self, x):
        return x


# hmm lot of things are going on here, but its all future work 







In [3]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
batch= []
txt1= "hare krishna hare krish"
txt2= "ohe vaishnav thakur"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch= torch.stack(batch,dim=0)

print(batch)

tensor([[43466,   479, 37518,  2616,   387,   260,   479, 37518],
        [   78,   258, 46935,   680, 28341,   294,   461,   333]])


In [4]:
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)
logits= model(batch)
print("Output shape:", logits.shape)
print(logits)
print(logits.shape)

Output shape: torch.Size([2, 8, 50257])
tensor([[[ 0.4754,  0.4974, -1.1567,  ..., -0.6499, -0.1118,  0.4062],
         [ 0.3593,  0.1491, -0.0736,  ...,  0.3925,  1.0802,  1.6621],
         [-0.8345,  0.9841,  0.5097,  ..., -0.0998,  0.4911, -0.4222],
         ...,
         [ 0.1697, -0.1739, -0.0934,  ...,  1.3824,  0.0409, -0.0772],
         [ 0.9715, -0.5424,  1.1583,  ..., -0.0080,  1.6833,  0.8957],
         [-0.7045,  0.9917,  0.2625,  ..., -0.6870, -1.1810,  0.0134]],

        [[-0.1740,  0.1334, -0.6599,  ...,  0.6796,  0.3747, -0.0669],
         [ 0.1848,  1.2332, -1.0756,  ...,  0.9119, -0.3607,  1.9177],
         [ 0.2497,  0.6356,  0.5857,  ...,  0.4055, -0.1022, -1.3354],
         ...,
         [-0.4680,  0.3159, -0.0927,  ...,  0.9989,  0.6678,  0.1298],
         [ 0.4681, -0.1927,  0.1978,  ...,  0.1749,  0.9242,  0.2378],
         [-0.0278,  1.3565, -0.3748,  ..., -0.4192, -0.5434, -1.0534]]],
       grad_fn=<UnsafeViewBackward0>)
torch.Size([2, 8, 50257])


lets learn layer norm

In [5]:
torch.manual_seed(123)
batch_example = torch.randn(2,5)
print(batch_example)
layer = nn.Sequential(nn.Linear(5,6), nn.ReLU())
out= layer(batch_example)
print(out)

tensor([[-0.1115,  0.1204, -0.3696, -0.2404, -1.1969],
        [ 0.2093, -0.9724, -0.7550,  0.3239, -0.1085]])
tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)


In [6]:
# lets examine the mean and variance
mean= out.mean(dim=-1, keepdim= True)
var= out.var(dim=-1, keepdim= True)
print("mean:\n", mean,"\n", "variance\n", var)

mean:
 tensor([[0.1324],
        [0.2170]], grad_fn=<MeanBackward1>) 
 variance
 tensor([[0.0231],
        [0.0398]], grad_fn=<VarBackward0>)


In [7]:
# lets do layer norm
out_norm = (out- mean)/torch.sqrt(var)
mean = out_norm.mean(dim=-1, keepdim= True)
var= out_norm.var(dim=-1, keepdim= True)
torch.set_printoptions(sci_mode= False)
print("layer_normalised\n", out_norm)
print("mean\n", mean)
print("variance\n",var)

layer_normalised
 tensor([[ 0.6159,  1.4126, -0.8719,  0.5872, -0.8719, -0.8719],
        [-0.0189,  0.1121, -1.0876,  1.5173,  0.5647, -1.0876]],
       grad_fn=<DivBackward0>)
mean
 tensor([[0.0000],
        [0.0000]], grad_fn=<MeanBackward1>)
variance
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


In [8]:
class LayerNorm(nn.Module):
    def __init__(self,emb_dim):
        super().__init__()
        self.eps= 1e-5
        self.scale= nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))
                                                                    
    def forward(self,x):
        mean= x.mean(dim=-1, keepdim= True)
        var= x.var(dim=-1, keepdim= True, unbiased= False)
        norm_x= (x-mean)/torch.sqrt(var + self.eps)
        return self.scale* norm_x + self.shift


In [9]:
ln= LayerNorm(emb_dim=5)
out_ln= ln(batch_example)

print(out_ln)

tensor([[ 0.5528,  1.0693, -0.0223,  0.2656, -1.8654],
        [ 0.9087, -1.3767, -0.9564,  1.1304,  0.2940]], grad_fn=<AddBackward0>)


In [10]:
mean= out_ln.mean(dim=-1, keepdim= True)
var = out_ln.var(dim =-1, unbiased= False, keepdim=True)
print("mean\n", mean)
print("variance\n", var)

mean
 tensor([[-0.0000],
        [ 0.0000]], grad_fn=<MeanBackward1>)
variance
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


In [11]:
import torch
import torch.nn as nn

class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

In [12]:
# import matplotlib.pyplot as plt
# import torch
# import torch.nn as nn

# gelu, relu = nn.GELU(), nn.ReLU()

# x = torch.linspace(-3, 3, 100)      # 100 points from -3 to 3

# y_gelu, y_relu = gelu(x), relu(x)

# plt.figure(figsize=(8, 3))

# for i, (y, label) in enumerate(
#         zip([y_gelu, y_relu], ["GELU", "ReLU"]), 1):

#     plt.subplot(1, 2, i)
#     plt.plot(x, y)
#     plt.title(f"{label} activation function")
#     plt.xlabel("x")
#     plt.ylabel(f"{label}(x)")
#     plt.grid(True)

# plt.tight_layout()
# plt.show()

In [13]:
1
class FeedForward(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.layers= nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
            GELU(),
            nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
        )
    def forward(self,x):
        return self.layers(X)

In [14]:
# hari_ffn= FeedForward(GPT_CONFIG_124M)
# x = torch.rand(2,3,768)
# out= hari_ffn(x)
# print(out.shape)


In [15]:
class ExampleDeepNeuralNetwork(nn.Module):
    def __init__(self, layer_size, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers= nn.ModuleList([
            nn.Sequential(nn.Linear(layer_size[0], layer_size[1]), 
                          GELU()),
            nn.Sequential(nn.Linear(layer_size[1], layer_size[2]), 
                          GELU()),
            nn.Sequential(nn.Linear(layer_size[2], layer_size[3]), 
                          GELU()),
            nn.Sequential(nn.Linear(layer_size[3], layer_size[4]), 
                          GELU()),
            nn.Sequential(nn.Linear(layer_size[4], layer_size[5]), 
                          GELU()),

        ])

    def forward(self,x):
        for layer in self.layers:
            layer_output= layer(x)
            if self.use_shortcut and x.shape == layer_output.shape:
                x = x+layer_output
            else:
                x= layer_output

        return x

In [16]:
x= torch.tensor([[1., 0., -1.]])
print(x.shape)
print(type(x))
print(x[-1].shape)
print("\n")
# y= torch.tensor([1., 0., -1.])
# print(y.shape)
# print(y[0], y[1])

torch.Size([1, 3])
<class 'torch.Tensor'>
torch.Size([3])




In [17]:
layer_sizes = [3,3,3,3,3,1]
sample_input = torch.tensor([[1., 0., -1.]])
torch.manual_seed(123)
model_without_shortcut = ExampleDeepNeuralNetwork(layer_sizes,
                                                  use_shortcut= False)

In [18]:
def print_gradients(model,x):
    output= model(x)
    target = torch.tensor([[0.]])

    loss= nn.MSELoss()
    loss = loss(output, target)
    
    loss.backward()

    for name, param in model.named_parameters():
        if 'weight' in name:
            print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")


In [19]:
model.named_parameters

<bound method Module.named_parameters of DummyGPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): DummyTransformerBlock()
    (1): DummyTransformerBlock()
    (2): DummyTransformerBlock()
    (3): DummyTransformerBlock()
    (4): DummyTransformerBlock()
    (5): DummyTransformerBlock()
    (6): DummyTransformerBlock()
    (7): DummyTransformerBlock()
    (8): DummyTransformerBlock()
    (9): DummyTransformerBlock()
    (10): DummyTransformerBlock()
    (11): DummyTransformerBlock()
  )
  (final_norm): DummyLayerNorm()
  (out_head): Linear(in_features=768, out_features=50257, bias=False)
)>

In [20]:
print_gradients(model_without_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.00020173587836325169
layers.1.0.weight has gradient mean of 0.0001201116101583466
layers.2.0.weight has gradient mean of 0.0007152041653171182
layers.3.0.weight has gradient mean of 0.001398873864673078
layers.4.0.weight has gradient mean of 0.005049646366387606


In [23]:
import import_ipynb
from attention import  MultiHeadAttention  # Imports my_other_notebook.ipynb from the same directory

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att= MultiHeadAttention(
            d_in = cfg["emb_dim"],
            d_out = cfg["emb_dim"],
            context_length= cfg["context_length"],
            num_heads= cfg["n_heads"],
            dropout= cfg["drop_rate"],
            qkv_bias = cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1= LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self,x):
        shortcut =x
        x= self.norm1(x)
        x= self.att(x)
        x= self.drop_shortcut(x)
        x= x+ shortcut

        shortcut =x
        x= self.norm2(x)
        x= self.att(x)
        x= self.drop_shortcut(x)
        x= x+ shortcut

        return x

In [24]:
torch.manual_seed(123)
x= torch.rand(2,4,768)
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

print("Input shape:" , x.shape)
print("output shape:" , output.shape)

AttributeError: 'Tensor' object has no attribute 'tranpose'